# Load model and dataset

In [1]:
! pip3 install torch --index-url https://download.pytorch.org/whl/

Looking in indexes: https://download.pytorch.org/whl/
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB 26.0 MB/s eta 0:01:14
   ---------------------------------------- 0.0/1.9 GB 17.5 MB/s eta 0:01:50
   ---------------------------------------- 0.0/1.9 GB 21.2 MB/s eta 0:01:30
    --------------------------------------- 0.0/1.9 GB 22.9 MB/s eta 0:01:23
    --------------------------------------- 0.0/1.9 GB 25.9 MB/s eta 0:01:13
    --------------------------------------- 0.0/1.9 GB 27.5 MB/s eta 0:01:09
   - ------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install pytorch_lightning nervaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4e56238e960479ead659822cd1d0de494bed3fd3c1ceb3efe593511f86701b65
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [6]:
!pip3 install -r "D:\univ_subj\M_An_I\Sem 2\LLMs for NLP\RoDi\performance_analysis\evaluate\requirements.txt" -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Evaluate.py

In [4]:
import os, json, torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoTokenizer, AutoModelForTokenClassification, set_seed
from pytorch_lightning.callbacks import EarlyStopping
from nervaluate import Evaluator
import pandas as pd

class TransformerModel(pl.LightningModule):
    def __init__(self, model_name, tokenizer, lr, lr_factor, lr_patience, model_max_length, bio2tags, tag_list):
        super().__init__()
        self.validation_step_outputs = []
        self.test_step_outputs = []

        print("Loading AutoModel [{}] ...".format(model_name))
        self.tokenizer = tokenizer
        self.model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(bio2tags), from_flax=False)

        self.lr = lr
        self.lr_factor = lr_factor
        self.lr_patience = lr_patience
        self.model_max_length = model_max_length
        self.bio2tags = bio2tags
        self.tag_list = tag_list
        self.num_labels = len(bio2tags)

        # add pad token
        self.validate_pad_token()

    def validate_pad_token(self):
        if self.tokenizer.pad_token is not None:
            return
        if self.tokenizer.sep_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the SEP token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.sep_token
            return
        if self.tokenizer.eos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the EOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.eos_token
            return
        if self.tokenizer.bos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the BOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.bos_token
            return
        if self.tokenizer.cls_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the CLS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.cls_token
            return
        raise Exception(
            "Could not detect SEP/EOS/BOS/CLS tokens, and thus could not assign a PAD token which is required.")

    def forward(self, input_ids, attention_mask, labels):
        output = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )
        return output["loss"], output["logits"]

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        loss, _ = self(input_ids, attention_mask, labels)
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        token_idx = batch["token_idx"]

        loss, logits = self(input_ids, attention_mask, labels)  # logits is [batch_size, seq_len, num_classes]

        batch_size = logits.size()[0]
        batch_pred = torch.argmax(logits.detach().cpu(), dim=-1).tolist()  # reduce to [batch_size, seq_len] as list
        batch_gold = labels.detach().cpu().tolist()  # [batch_size, seq_len] as list
        batch_token_idx = token_idx.detach().cpu().tolist()

        for batch_idx in range(batch_size):
            pred, gold, idx = batch_pred[batch_idx], batch_gold[batch_idx], batch_token_idx[batch_idx]
            y_hat, y = [], []
            for i in range(0, max(idx) + 1): # for each sentence
                pos = idx.index(i)  # find next token index and get pred and gold
                y_hat.append(pred[pos])
                y.append(gold[pos])

        self.validation_step_outputs.append({
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        })

        return {
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        }

    def on_validation_epoch_end(self):
        odf = pd.DataFrame(self.validation_step_outputs)

        mean_val_loss = odf["loss"].mean()
        gold, pred = [], []
        for _, row in odf.iterrows():
            gold.append([self.bio2tags[token_id] for token_id in row["y"]])
            pred.append([self.bio2tags[token_id] for token_id in row["y_hat"]])

        evaluator = Evaluator(gold, pred, tags=self.tag_list, loader="list")

        results, _ = evaluator.evaluate()
        self.log("valid/avg_loss", mean_val_loss, prog_bar=True)
        self.log("valid/ent_type", float(results["ent_type"]["f1"]))
        self.log("valid/partial", float(results["partial"]["f1"]))
        self.log("valid/strict", float(results["strict"]["f1"]), prog_bar=True)
        self.log("valid/exact", float(results["exact"]["f1"]))

        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        token_idx = batch["token_idx"]

        loss, logits = self(input_ids, attention_mask, labels)  # logits is [batch_size, seq_len, num_classes]

        batch_size = logits.size()[0]
        batch_pred = torch.argmax(logits.detach().cpu(), dim=-1).tolist()  # reduce to [batch_size, seq_len] as list
        batch_gold = labels.detach().cpu().tolist()  # [batch_size, seq_len] as list
        batch_token_idx = token_idx.detach().cpu().tolist()

        for batch_idx in range(batch_size):
            pred, gold, idx = batch_pred[batch_idx], batch_gold[batch_idx], batch_token_idx[batch_idx]
            y_hat, y = [], []
            for i in range(0, max(idx) + 1):  # for each sentence
                pos = idx.index(i)  # find next token index and get pred and gold
                y_hat.append(pred[pos])
                y.append(gold[pos])

        self.test_step_outputs.append({
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        })

        return {
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        }

    def on_test_epoch_end(self):
        odf = pd.DataFrame(self.test_step_outputs)

        mean_val_loss = odf["loss"].mean()
        gold, pred = [], []
        for _, row in odf.iterrows():
            gold.append([self.bio2tags[token_id] for token_id in row["y"]])
            pred.append([self.bio2tags[token_id] for token_id in row["y_hat"]])

        evaluator = Evaluator(gold, pred, tags=self.tag_list, loader="list")

        results, _ = evaluator.evaluate()
        self.log("test/avg_loss", mean_val_loss, prog_bar=True)
        self.log("test/ent_type", results["ent_type"]["f1"])
        self.log("test/partial", results["partial"]["f1"])
        self.log("test/strict", results["strict"]["f1"])
        self.log("test/exact", results["exact"]["f1"])

        self.test_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW([p for p in self.parameters() if p.requires_grad], lr=self.lr, eps=1e-08)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    factor=self.lr_factor,
                    patience=self.lr_patience,
                    mode='max'
                ),
                'interval': 'epoch',
                'frequency': 1,
                'monitor': 'valid/strict',
                'strict': True,
                'name': 'learning_rate',
            },
        }

    def predict(self, input_string):
        input_ids = self.tokenizer.encode(input_string, add_special_tokens=False)

        # run the model
        output = self.model(input_ids=torch.unsqueeze(torch.LongTensor(input_ids), 0), return_dict=True)
        logits = output["logits"]

        # extract results
        indices = torch.argmax(logits.detach().cpu(), dim=-1).squeeze(dim=0).tolist()  # reduce to [batch_size, seq_len] as list

        output_ids = []

        for id in input_ids:
            output_ids.append(self.tokenizer.decode(id))

        output_classes = []
        for i in indices:
            output_classes.append(self.bio2tags[i])

        return output_ids, output_classes


class RoNecDataset(Dataset):
    def __init__(self, instances):
        self.instances = []

        # run check
        for instance in instances:
            ok = True
            if len(instance["ner_ids"]) != len(instance["tokens"]):
                print("Different length ner_tags found")
                ok = False
            else:
                for _, token in zip(instance["ner_ids"], instance["tokens"]):
                    if token.strip() == "":
                        ok = False
                        print("Empty token found")
            if ok:
                self.instances.append(instance)

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, i):
        return self.instances[i]


class Collator(object):
    def __init__(self, tokenizer, max_seq_len):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

        self.validate_pad_token()

    def validate_pad_token(self):
        if self.tokenizer.pad_token is not None:
            return
        if self.tokenizer.sep_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the SEP token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.sep_token
            return
        if self.tokenizer.eos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the EOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.eos_token
            return
        if self.tokenizer.bos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the BOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.bos_token
            return
        if self.tokenizer.cls_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the CLS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.cls_token
            return
        raise Exception("Could not detect SEP/EOS/BOS/CLS tokens, and thus could not assign a PAD token which is required.")


    def __call__(self, input_batch):
        batch_input_ids, batch_labels, batch_attention, batch_token_idx = [], [], [], []
        max_len = 0

        for instance in input_batch:
            instance_ids, instance_labels, instance_attention, instance_token_idx = [], [], [], []

            for i in range(len(instance["tokens"])):
                subids = self.tokenizer.encode(instance["tokens"][i], add_special_tokens=False)
                sublabels = [instance["ner_ids"][i]]

                if len(subids) > 1:  # we have a word split in more than 1 subids, fill appropriately
                    filler_sublabel = sublabels[0] if sublabels[0] % 2 == 0 else sublabels[0] + 1
                    sublabels.extend([filler_sublabel] * (len(subids) - 1))

                instance_ids.extend(subids)  # extend with the number of subids
                instance_labels.extend(sublabels)  # extend with the number of subtags
                instance_token_idx.extend([i] * len(subids))  # extend with the id of the token

                assert len(subids) == len(sublabels) # check for possible errors in the dataset

            if len(instance_ids) != len(instance_labels):
                print(len(instance_ids))
                print(len(instance_labels))
                print(instance_ids)
                print(instance_labels)
            assert len(instance_ids) == len(instance_labels)

            # cut to max sequence length, if needed
            if len(instance_ids) > self.max_seq_len - 2:
                instance_ids = instance_ids[:self.max_seq_len - 2]
                instance_labels = instance_labels[:self.max_seq_len - 2]
                instance_token_idx = instance_token_idx[:self.max_seq_len - 2]

            # prepend and append special tokens, if needed
            if self.tokenizer.cls_token_id and self.tokenizer.sep_token_id:
                instance_ids = [self.tokenizer.cls_token_id] + instance_ids + [self.tokenizer.sep_token_id]
                instance_labels = [0] + instance_labels + [0]
                instance_token_idx = [-1] + instance_token_idx  # no need to pad the last, will do so automatically at return
            instance_attention = [1] * len(instance_ids)

            # update max_len for later padding
            max_len = max(max_len, len(instance_ids))

            # add to batch
            batch_input_ids.append(torch.LongTensor(instance_ids))
            batch_labels.append(torch.LongTensor(instance_labels))
            batch_attention.append(torch.LongTensor(instance_attention))
            batch_token_idx.append(torch.LongTensor(instance_token_idx))

        return {
            "input_ids": torch.nn.utils.rnn.pad_sequence(batch_input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else 0),
            "attention_mask": torch.nn.utils.rnn.pad_sequence(batch_attention, batch_first=True, padding_value=0),
            "labels": torch.nn.utils.rnn.pad_sequence(batch_labels, batch_first=True, padding_value=0),
            "token_idx": torch.nn.utils.rnn.pad_sequence(batch_token_idx, batch_first=True, padding_value=-1)
        }


def run_evaluation(args):
    print("Loading data...")
    with open(args.train_file, "r", encoding="utf8") as f:
        train_data = json.load(f)
    with open(args.validation_file, "r", encoding="utf8") as f:
        validation_data = json.load(f)
    with open(args.test_file, "r", encoding="utf8") as f:
        test_data = json.load(f)

    # deduce bio2 tag mapping and simple tag list, required by nervaluate
    # deduce bio2 tag mapping and simple tag list, required by nervaluate
    tags = ["O"] * 16  # tags without the B- or I- prefix
    bio2tags = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

    for instance in train_data:
        for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
            bio2tags[tag_index] = tag  # put the bio2 tag in its correct position
            if tag_index % 2 == 0 and tag_index > 0:
                tags[int(tag_index / 2)] = tag[2:]

    print(f"\tDataset contains {len(bio2tags)} BIO2 classes: {bio2tags}.")
    print(f"\tThere are {len(tags)} classes: {tags}\n")

    # init tokenizer and start loading data
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, strip_accents=False)
    train_dataset = RoNecDataset(train_data)
    val_dataset = RoNecDataset(validation_data)
    test_dataset = RoNecDataset(test_data)

    collator = Collator(tokenizer=tokenizer, max_seq_len=args.model_max_length)

    train_dataloader = DataLoader(train_dataset, batch_size=args.batch_size, num_workers=4, shuffle=True,
                                  collate_fn=collator, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, num_workers=4, shuffle=False,
                                collate_fn=collator, pin_memory=True)
    test_dataloader = DataLoader(test_dataset, batch_size=args.batch_size, num_workers=4, shuffle=False,
                                 collate_fn=collator, pin_memory=True)

    print("\tTrain dataset has {} instances.".format(len(train_dataset)))
    print("\tValid dataset has {} instances.".format(len(val_dataset)))
    print("\tTest dataset has {} instances.\n".format(len(test_dataset)))

    itt = 0

    valid_loss = []
    valid_ent_type = []
    valid_partial = []
    valid_strict = []
    valid_exact = []
    test_loss = []
    test_ent_type = []
    test_partial = []
    test_strict = []
    test_exact = []
    while itt < args.experiment_iterations:
        print("Running experiment {}/{}".format(itt + 1, args.experiment_iterations))

        model = TransformerModel(
            model_name=args.model_name,
            lr=args.lr,
            lr_factor=args.lr_factor,
            lr_patience=args.lr_patience,
            model_max_length=args.model_max_length,
            bio2tags=bio2tags,
            tokenizer=tokenizer,
            tag_list=tags,
        )

        early_stop = EarlyStopping(
            monitor='valid/strict',
            min_delta=0.0001,
            patience=5,
            verbose=True,
            mode='max'
        )

        lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval='epoch')

        model_checkpointer = pl.callbacks.ModelCheckpoint(save_top_k=1, monitor='valid/strict', dirpath=args.dirpath,
                                                          filename='{epoch}', mode='max', save_on_train_epoch_end=True)

        trainer = pl.Trainer(
            accelerator='gpu',
            devices=args.devices,
            strategy=args.strategy,
            max_epochs=args.max_epochs,
            callbacks=[lr_monitor, early_stop, model_checkpointer],
            accumulate_grad_batches=args.accumulate_grad_batches,
            gradient_clip_val=1.0,
            #limit_train_batches=50,
            #limit_val_batches=50,
        )
        trainer.fit(model, train_dataloader, val_dataloader)

        print("\nEvaluating model on the VALIDATION dataset:")
        result_valid = trainer.test(model, val_dataloader)
        print("\nEvaluating model on the TEST dataset:")
        result_test = trainer.test(model, test_dataloader)

        with open("results_ronec_{}_of_{}.json".format(itt + 1, args.experiment_iterations), "w") as f:
            json.dump(result_test[0], f, indent=4, sort_keys=True)

        valid_loss.append(result_valid[0]['test/avg_loss'])
        valid_ent_type.append(result_valid[0]['test/ent_type'])
        valid_partial.append(result_valid[0]['test/partial'])
        valid_strict.append(result_valid[0]['test/strict'])
        valid_exact.append(result_valid[0]['test/exact'])
        test_loss.append(result_test[0]['test/avg_loss'])
        test_ent_type.append(result_test[0]['test/ent_type'])
        test_partial.append(result_test[0]['test/partial'])
        test_strict.append(result_test[0]['test/strict'])
        test_exact.append(result_test[0]['test/exact'])

        itt += 1

    print("Done, writing results...\n")

    result = {
        "valid_loss": sum(valid_loss) / args.experiment_iterations,
        "valid_ent_type": sum(valid_ent_type) / args.experiment_iterations,
        "valid_partial": sum(valid_partial) / args.experiment_iterations,
        "valid_strict": sum(valid_strict) / args.experiment_iterations,
        "valid_exact": sum(valid_exact) / args.experiment_iterations,
        "test_loss": sum(test_loss) / args.experiment_iterations,
        "test_ent_type": sum(test_ent_type) / args.experiment_iterations,
        "test_partial": sum(test_partial) / args.experiment_iterations,
        "test_strict": sum(test_strict) / args.experiment_iterations,
        "test_exact": sum(test_exact) / args.experiment_iterations
    }

    with open("results_{}.json".format(args.model_name.replace("/", "_")), "w") as f:
        json.dump(result, f, indent=4, sort_keys=True)

    print("\nFinal averaged results on TEST data: ")
    from pprint import pprint
    pprint(result)


# if __name__ == "__main__":
#     from argparse import ArgumentParser

#     parser = ArgumentParser()
#     parser.add_argument('--seed', type=int, default=-1)
#     parser.add_argument('--max_epochs', type=int, default=1000)
#     parser.add_argument('--batch_size', type=int, default=8)
#     parser.add_argument('--accumulate_grad_batches', type=int, default=2)
#     parser.add_argument('--model_name', type=str, default="dumitrescustefan/bert-base-romanian-uncased-v1")
#     parser.add_argument("--train_file", type=str, default="../data/train.json")
#     parser.add_argument("--validation_file", type=str, default="../data/valid.json")
#     parser.add_argument("--test_file", type=str, default="../data/test.json")
#     parser.add_argument("--dirpath", type=str, default=None)
#     parser.add_argument('--lr', type=float, default=2e-05)
#     parser.add_argument('--lr_factor', type=float, default=2/3)
#     parser.add_argument('--lr_patience', type=float, default=5)
#     parser.add_argument('--model_max_length', type=int, default=512)
#     parser.add_argument('--experiment_iterations', type=int, default=1)
#     parser.add_argument('--devices', type=int, default=1)
#     parser.add_argument('--strategy', type=str, default=None)

#     args = parser.parse_args()

#     if args.seed >= 0:
#         pl.seed_everything(args.seed, workers=True)
#         set_seed(args.seed)
#         os.environ['PYTHONHASHSEED']=str(args.seed)
#     else:
#         print("Using a random seed.")

#     run_evaluation(args)


In [5]:
import torch
import json
import numpy as np
# from seqeval.metrics import classification_report

# ── Load best checkpoint ────────────────────────────────────
checkpoints_path = './performance_analysis/checkpoints/'
datasets_path = './performance_analysis/datasets/'
m_model_name = 'bert-base-multilingual-cased'
ro_model_name = 'dumitrescustefan/bert-base-romanian-cased-v1'

In [ ]:
with open(datasets_path + 'diac/train.json', "r", encoding="utf8") as f:
    train_diac = json.load(f)

In [ ]:
tags_diac = ["O"] * 16  # tags without the B- or I- prefix
bio2tags_diac = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

for instance in train_diac:
    for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
        bio2tags_diac[tag_index] = tag  # put the bio2 tag in its correct position
        if tag_index % 2 == 0 and tag_index > 0:
            tags_diac[int(tag_index / 2)] = tag[2:]

In [5]:
best_ckpt = checkpoints_path + 'm_diac/epoch=2.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_diac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=m_model_name,        # e.g., "bert-base-multilingual-cased"
    tokenizer=AutoTokenizer.from_pretrained(m_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=your_bio2tags_dict,
    tag_list=your_tag_list
)
m_diac_model.eval()
m_diac_model.cuda()

NameError: name 'checkpoints_path' is not defined

In [ ]:

best_ckpt = checkpoints_path + 'm_nodiac/epoch=3_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_nodiac_model = TransformerModel.load_from_checkpoint(best_ckpt)
m_nodiac_model.eval()
m_nodiac_model.cuda()

In [ ]:
best_ckpt = checkpoints_path + 'ro_diac/epoch=1_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_diac_model = TransformerModel.load_from_checkpoint(best_ckpt)
ro_diac_model.eval()
ro_diac_model.cuda()

In [ ]:
best_ckpt = checkpoints_path + 'ro_nodiac/epoch=5_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_nodiac_model = TransformerModel.load_from_checkpoint(best_ckpt)
ro_nodiac_model.eval()
ro_nodiac_model.cuda()

# Crossed Inference

In [ ]:
# Configs
batch_size = 8

# "robert" | "mbert"
MODEL_KEY  = "mbert"
# "diac" | "nodiac"
TRAIN_COND = "nodiac"

RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

MODEL_HUB = {
    "robert": "dumitrescustefan/bert-base-romanian-cased-v1",
    "mbert":  "bert-base-multilingual-cased",
}[MODEL_KEY]

# SEED = 315
SEED = 222

BASE     = f"/content/drive/MyDrive/rodi_study/{RUN_NAME}"
CKPT_DIR = f"{BASE}/checkpoints"
RES_DIR  = f"{BASE}/results"
PRED_DIR = f"{BASE}/predictions"


In [ ]:
label_list = ['PERSON', 'GPE', 'LOC', 'ORG', 'LANGUAGE', 'NAT_REL_POL', 'DATETIME', 'PERIOD', 'QUANTITY', 'MONEY', 'NUMERIC', 'ORDINAL', 'FACILITY', 'WORK_OF_ART', 'EVENT']

In [ ]:
def run_inference(model, dataloader, device="cuda"):
    """
    Mirrors the evaluate script's validation_step logic but
    stores per-sentence results instead of aggregating them.
    Returns a list of dicts: {tokens, gold_tags, pred_tags}
    """
    model.eval()
    records = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids, attention_mask, labels, batch_token_idx = batch
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            logits = model(input_ids, attention_mask)   # (B, seq_len, num_labels)
            preds  = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.cpu().numpy()

            for sent_preds, sent_labels, token_idx in zip(preds, labels, batch_token_idx):
                # token_idx: list of subword positions for each word
                word_preds  = [sent_preds[idx]  for idx in token_idx]
                word_labels = [sent_labels[idx]  for idx in token_idx]

                # Decode integer ids → tag strings
                pred_tags = [label_list[i] for i in word_preds]
                gold_tags = [label_list[i] for i in word_labels]

                records.append({
                    "pred_tags": pred_tags,
                    "gold_tags": gold_tags,
                })

    return records

In [ ]:
with open(datasets_path + 'diac/test.json', "r", encoding="utf8") as f:
    test_diac = json.load(f)
with open(datasets_path + 'nodiac/test.json', "r", encoding="utf8") as f:
    test_nodiac = json.load(f)

In [ ]:
def eval_model(model_name, model):
    for eval_cond, test_dataset in [("diac", test_diac), ("nodiac", test_nodiac)]:
        tokenizer = AutoTokenizer.from_pretrained(model_name, strip_accents=False)
        collator = Collator(tokenizer=tokenizer, max_seq_len=args.model_max_length)

        test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                                     collate_fn=collator, pin_memory=True)


        records = run_inference(model, test_loader)

        # ── Save raw predictions ────────────────────────────────
        pred_path = f"{PRED_DIR}/{RUN_NAME}_eval_{eval_cond}.json"
        with open(pred_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        # ── Per-class seqeval report ────────────────────────────
        golds = [r["gold_tags"] for r in records]
        preds = [r["pred_tags"] for r in records]

        report = classification_report(golds, preds, output_dict=True, zero_division=0)
        import pandas as pd
        df = pd.DataFrame(report).T
        df.to_csv(f"{RES_DIR}/{RUN_NAME}_eval_{eval_cond}_per_class.csv")
        print(f"\n=== {RUN_NAME} → eval {eval_cond} ===")
        print(df.to_string())
        return df, records

In [ ]:
rows = []

df, m_diac_records = eval_model(m_model_name, m_diac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'm_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})


df, m_nodiac_records = eval_model(m_model_name, m_nodiac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'm_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})


df, ro_diac_records = eval_model(ro_model_name, ro_diac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'ro_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

df, ro_nodiac_records = eval_model(ro_model_name, ro_nodiac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'ro_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

summary = (pd.DataFrame(rows)
             .pivot_table(index="run", columns="eval", values=["P","R","F1"]))
print(summary.to_markdown())
summary.to_csv("4x4_matrix.csv")

# Error analysis

In [ ]:
def categorize_errors(records, sent_offset=0):
    """
    Takes the records list from run_inference (gold_tags, pred_tags per sentence).
    Returns a DataFrame with one row per span-level error.
    error_type ∈ {missed, spurious, wrong_type, wrong_boundary}
    """
    import pandas as pd

    def get_spans(tags):
        spans, i = {}, 0
        while i < len(tags):
            if tags[i].startswith("B-"):
                label = tags[i][2:]
                j = i + 1
                while j < len(tags) and tags[j] == f"I-{label}":
                    j += 1
                spans[(i, j - 1)] = label
                i = j
            else:
                i += 1
        return spans

    rows = []
    for sid, rec in enumerate(records, start=sent_offset):
        gold_spans = get_spans(rec["gold_tags"])
        pred_spans = get_spans(rec["pred_tags"])

        done_gold, done_pred = set(), set()

        # Wrong type: same (start, end), different label
        for pos in set(gold_spans) & set(pred_spans):
            gl, pl = gold_spans[pos], pred_spans[pos]
            if gl != pl:
                rows.append(dict(sent_id=sid, error_type="wrong_type",
                                 gold_class=gl, pred_class=pl,
                                 gold_span=pos, pred_span=pos))
                done_gold.add(pos); done_pred.add(pos)

        # Wrong boundary: overlapping spans with same label
        for gpos, gl in gold_spans.items():
            if gpos in done_gold: continue
            for ppos, pl in pred_spans.items():
                if ppos in done_pred: continue
                gs, ge = gpos; ps, pe = ppos
                if gl == pl and gs <= pe and ps <= ge and gpos != ppos:
                    rows.append(dict(sent_id=sid, error_type="wrong_boundary",
                                     gold_class=gl, pred_class=pl,
                                     gold_span=gpos, pred_span=ppos))
                    done_gold.add(gpos); done_pred.add(ppos); break

        # Missed (false negatives)
        for pos, label in gold_spans.items():
            if pos not in done_gold:
                rows.append(dict(sent_id=sid, error_type="missed",
                                 gold_class=label, pred_class="O",
                                 gold_span=pos, pred_span=None))

        # Spurious (false positives)
        for pos, label in pred_spans.items():
            if pos not in done_pred:
                rows.append(dict(sent_id=sid, error_type="spurious",
                                 gold_class="O", pred_class=label,
                                 gold_span=None, pred_span=pos))

    return pd.DataFrame(rows)


for records, eval_cond, run_name in zip([m_diac_records, m_nodiac_records, ro_diac_records, ro_nodiac_records], ['diac', 'nodiac', 'diac', 'nodiac'], ['m_diac', 'm_nodiac', 'ro_diac', 'ro_nodiac']):

    errors = categorize_errors(records)
    errors.to_csv(f"{RES_DIR}/{run_name}_eval_{eval_cond}_errors.csv", index=False)

    print(f"\n--- {eval_cond} error breakdown ---")
    print(errors.groupby(["gold_class", "error_type"]).size().unstack(fill_value=0))

# Attention analysis

In [ ]:
def extract_attentions(model, tokenizer, sentence, device="cuda"):
    """
    Returns (token_strings, attentions_ndarray).
    attentions shape: (num_layers, num_heads, seq_len, seq_len)
    """
    model.eval()
    enc = tokenizer(sentence, return_tensors="pt",
                    truncation=True, max_length=512).to(device)

    with torch.no_grad():
        # Pass output_attentions=True directly to the underlying BERT:
        out = model.bert(**enc, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    attns  = np.stack([a.squeeze(0).cpu().numpy() for a in out.attentions])
    return tokens, attns  # (num_layers, num_heads, seq, seq)


configuration = [
    {
        'records': m_diac_records,
        'eval_cond': 'diac',
        'run_name': 'm_diac',
        'model': m_diac_model,
        'model_name': m_model_name,
    },
    {
        'records': m_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'm_nodiac',
        'model': m_nodiac_model,
        'model_name': m_model_name,
    },
    {
        'records': ro_diac_records,
        'eval_cond': 'diac',
        'run_name': 'ro_diac',
        'model': ro_diac_model,
        'model_name': ro_model_name,
    },
    {
        'records': ro_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'ro_nodiac',
        'model': ro_nodiac_model,
        'model_name': ro_model_name,
    }
]

for config in configuration:
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], strip_accents=False)

    # Save attention for the first N missed entities from the diac eval:
    errors = pd.read_csv(f"{RES_DIR}/{config['run_name']}_eval_errors.csv")

    sample_ids = errors[errors["error_type"] == "missed"]["sent_id"].unique()[:30]

    for sid in sample_ids:
        rec     = config['records'][int(sid)]
        sentence = " ".join(rec["gold_tags"])  # or reconstruct from raw dataset
        tokens, attns = extract_attentions(config['model'], tokenizer, sentence)
        np.savez_compressed(
            f"{RES_DIR}/attn_sent{sid}_{config['run_name']}.npz",
            attns=attns,
            tokens=np.array(tokens, dtype=object),
        )